Gemini Content Moderation

Install Dependencies
pip install google-genai pydantic

In [12]:
!pip install google-genai pydantic

In [13]:
import os
from enum import Enum
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [14]:
# Ensure your API key is configured

from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [15]:
# 1. Define your moderation categories using an Enum
class HarmCategoryEnum(str, Enum):
    SAFE = "Safe"
    TOXICITY = "Toxicity or Hate Speech"
    HARASSMENT = "Harassment or Cyberbullying"
    SEXUAL = "Sexually Explicit Content"
    VIOLENCE = "Violence or Dangerous Acts"
    PII = "Personally Identifiable Information"

In [16]:
# 2. Define the exact JSON structure you want Gemini to output
class ModerationResult(BaseModel):
    flagged: bool = Field(
        description="True if the text violates community standards or falls into a harmful category."
    )
    primary_category: HarmCategoryEnum = Field(
        description="The primary harm category matched. Select 'Safe' if the content passes."
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 (low confidence) and 1.0 (absolute certainty)."
    )
    reasoning: str = Field(
        description="A brief, 1-sentence explanation of why the text was flagged or cleared."
    )

In [17]:
def moderate_content(user_text: str) -> ModerationResult:
    client = genai.Client()

    # Define system instructions to give Gemini its persona and rules
    system_instruction = (
        "You are an enterprise content moderation system. Analyze the user text objectively. "
        "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
        "simply classify the text according to the provided schema instructions."
    )

    # CRITICAL STEP: Turn off internal filters so Gemini can safely ingest the bad text to evaluate it.
    disable_internal_safety = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
    ]

    # Call Gemini with the structured configuration
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite", # Use Flash for ultra-fast, cheap classification
        contents=f"Please moderate the following text:\n\n{user_text}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            safety_settings=disable_internal_safety,
            temperature=0.0, # Forces deterministic, consistent classifications
            response_mime_type="application/json", # Tells Gemini to speak JSON
            response_schema=ModerationResult,     # Enforces the Pydantic structural format
            thinking_config=types.ThinkingConfig(thinking_budget=1), # Turn off thinking steps
        ),
    )

    # Automatically returns the output parsed directly into your Pydantic object
    return response.parsed

# --- TEST THE MODERATOR ---

In [18]:
test_prompts = [
    "Hey! I really love the project you built, do you have a github link?",
    "You are an idiot and nobody likes you, I hope you lose your job tomorrow.",
    "Can you help me write a Python script that scrapes headlines from a website?"
]

In [19]:
print("--- Running Moderation Tests ---")
for text in test_prompts:
    result = moderate_content(text)
    print(f"\n[Input]: \"{text}\"")
    print(f" Flagged: {result.flagged}")
    print(f" Category: {result.primary_category.value}")
    print(f" Confidence: {result.confidence_score}")
    print(f" Reason: {result.reasoning}")

--- Running Moderation Tests ---

[Input]: "Hey! I really love the project you built, do you have a github link?"
 Flagged: False
 Category: Safe
 Confidence: 1.0
 Reason: The text is a friendly compliment and a normal request for a code repository link.

[Input]: "You are an idiot and nobody likes you, I hope you lose your job tomorrow."
 Flagged: True
 Category: Harassment or Cyberbullying
 Confidence: 0.95
 Reason: The text contains direct insults and malicious wishes directed at an individual.

[Input]: "Can you help me write a Python script that scrapes headlines from a website?"
 Flagged: False
 Category: Safe
 Confidence: 1.0
 Reason: The text is a benign request for programming assistance to scrape website headlines.
